In [10]:
!pip install datafast

"pip" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


In [11]:
from datafast.datasets import ClassificationDataset
from datafast.schema.config import ClassificationDatasetConfig, PromptExpansionConfig
from datafast.llms import OpenAIProvider, AnthropicProvider, GeminiProvider, OpenRouterProvider, OllamaProvider
from datafast.logger_config import configure_logger

# Load environment variables
#load_dotenv() # <--- your API keys
# Configure logger for visibility into generation process

configure_logger() # <--- see progress, warnings, and success messages


config = ClassificationDatasetConfig(
    classes=[
        # 1. ORDENADOR DE A BORDO (TRISKEL)
        {
            "name": "GET_TEMP", "description": "Inquiries about the current temperature of the satellite or the TRISKEL OBC. Covers generic 'how warm are you' questions."
        },
        {
            "name": "TRISKEL_GET_CURRENT", "description": "Requests for the specific amperage or current consumption of the TRISKEL on-board computer (OBC)."
        },
        
        # 2. PARÁMETROS ORBITALES (ELEMENTOS KEPLERIANOS)
        {
            "name": "ORBIT_GET_ALT", "description": "Requests for numerical values of altitude or the semi-major axis defining the orbit size."
        },
        {
            "name": "ORBIT_GET_ECCENTRICITY", "description": "Inquiries about the orbital eccentricity (0 for circular, close to 1 for elliptical)."
        },
        {
            "name": "ORBIT_GET_INCLINATION", "description": "Questions about the orbital inclination in degrees relative to the equator."
        },
        {
            "name": "ORBIT_GET_RAAN", "description": "Requests for the Right Ascension of the Ascending Node (RAAN), defining orbital orientation."
        },
        {
            "name": "ORBIT_GET_PERIGEE", "description": "Inquiries about the Argument of Perigee, the point in the orbit closest to Earth."
        },
        {
            "name": "ORBIT_GET_TRUE_ANOMALY", "description": "Requests for the True Anomaly, the actual real-time angular position of the satellite."
        },
        {
            "name": "ORBIT_GET_MEAN_ANOMALY", "description": "Inquiries about the Mean Anomaly, the averaged position used for orbital calculations."
        },

        # 3. SISTEMA DE POTENCIA (ICEPS2)
        {
            "name": "POWER_GET_SOC", "description": "Questions about the battery State of Charge (SoC) or remaining battery percentage."
        },
        {
            "name": "POWER_GET_DOD", "description": "Inquiries about the Depth of Discharge (DoD) or how much battery energy has been used."
        },
    ],

    num_samples_per_prompt=11,
    output_file="estigia_dataset.jsonl",
    languages={"en": "English"},  # Añadido español ya que tus ejemplos estaban en ese idioma
    prompts=[(
        "Generate {num_samples} human-like questions in {language_name} "
        "for the class '{label_name}'. Context: {label_description}. "
        "Imagine you are a student casually talking to the Estigia satellite. "
        "Use a {{style}} style: friendly, casual, natural, like a chat message. "
        "Do NOT include any JSON, labels, or structured data. "
        "Just write the question exactly as a human would ask it."
    )],

    expansion=PromptExpansionConfig(
        placeholders={
            "style": ["brief", "natural", "casual", "friendly"]
        },
        combinatorial=True
    ),
    shuffle=True
)

#providers = [OllamaProvider(model_id="llama3.1:latest", api_base="http://localhost:11434")]
providers = [OllamaProvider(model_id="qwen3:14b", api_base="http://PLUTON-UPV:11434"),
              OllamaProvider(model_id="gpt-oss", api_base="http://PLUTON-UPV:11434"),
              OllamaProvider(model_id="llama3.1:8b", api_base="http://PLUTON-UPV:11434")]
#providers = [OllamaProvider(model_id="gemma:2b", api_base="http://PLUTON-UPV:11434")]

# Generate dataset and local save

dataset = ClassificationDataset(config)

dataset.generate(providers)


2026-02-19 18:00:45 | INFO     | datafast.llms:__init__ - Initialized ollama_chat | Model: qwen3:14b
2026-02-19 18:00:45 | INFO     | datafast.llms:__init__ - Initialized ollama_chat | Model: gpt-oss
2026-02-19 18:00:45 | INFO     | datafast.llms:__init__ - Initialized ollama_chat | Model: llama3.1:8b
2026-02-19 18:00:45 | INFO     | datafast.datasets:generate - Starting ClassificationDataset.generate() | Expected rows: 1452 | Providers: 3
2026-02-19 18:01:44 | SUCCESS  | datafast.utils:log_generation_progress - Generated and saved 11 examples total | Provider: ollama_chat | Model: qwen3:14b | Duration: 59.0s
2026-02-19 18:02:12 | SUCCESS  | datafast.utils:log_generation_progress - Generated and saved 22 examples total | Provider: ollama_chat | Model: gpt-oss | Duration: 27.5s
2026-02-19 18:02:22 | SUCCESS  | datafast.utils:log_generation_progress - Generated and saved 33 examples total | Provider: ollama_chat | Model: llama3.1:8b | Duration: 9.9s
2026-02-19 18:02:38 | SUCCESS  | dataf

In [ ]:
#Formateo del dataset
import json

def transformar_dataset(archivo_entrada, archivo_salida):
    dataset_final = []
    
    with open(archivo_entrada, 'r', encoding='utf-8') as f:
        for linea in f:
            if not linea.strip():
                continue
            
            # Cargar el objeto original
            data_original = json.loads(linea)
            
            # Mapear al nuevo formato de columnas
            nuevo_objeto = {
                "Question": data_original.get("text"),
                "Label": data_original.get("label"),
                "Language": data_original.get("language")
            }
            dataset_final.append(nuevo_objeto)
    
    # Guardar como JSON estructurado (lista de objetos)
    with open(archivo_salida, 'w', encoding='utf-8') as f:
        json.dump(dataset_final, f, ensure_ascii=False, indent=4)

In [ ]:
# Uso del script
transformar_dataset("estigia_dataset.jsonl", "estigia_dataset_formateada.json")

In [1]:
#Función para subir el dataset a HuggingFace
from huggingface_hub import login
login(token="YOUR_HF_TOKEN")
from datasets import Dataset
ds = Dataset.from_json("estigia_dataset_formateada.json")
ds.push_to_hub("plutonupv/Estigia_telemetry")

c:\Users\USUARIO\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 151100 examples [00:00, 251463.86 examples/s]
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00,  4.72ba/s]
Processing Files (1 / 1): 100%|██████████| 31.5MB / 31.5MB, 7.88MB/s  
New Data Upload: 100%|██████████| 30.0MB / 30.0MB, 7.51MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:05<00:00,  5.36s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/plutonupv/Estigia_telemetry/commit/c0a6079b4370d8a5c75bdaa79a68fb4a95a5a848', commit_message='Upload dataset', commit_description='', oid='c0a6079b4370d8a5c75bdaa79a68fb4a95a5a848', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/plutonupv/Estigia_telemetry', endpoint='https://huggingface.co', repo_type='dataset', repo_id='plutonupv/Estigia_telemetry'), pr_revision=None, pr_num=None)